In [2]:
import random
from collections import Counter

# ----------------------------
# トランプ
# ----------------------------
class t_cards:  #山札を管理するクラス
    def __init__(self): #トランプのカードを初期化
        self.symbol = ['♠', '♥', '♦', '♣']
        self.number = ['A', '2', '3', '4', '5', '6', '7', '8',
                       '9', '10', 'J', 'Q', 'K']
        #52枚のカードを生成
        self.trump_card = [s + n for s in self.symbol for n in self.number]
        #カードをシャッフル
        random.shuffle(self.trump_card)

    #カードを引く
    def draw(self):
        return self.trump_card.pop()


# ----------------------------
# プレイヤー（人間＆親(cpu)共通）
# ----------------------------
class player:
    def __init__(self, all_cards):
        self.all_cards = all_cards
        self.selected_cards = []  #現在の手札

    # ★ 5枚引く
    def card_selection(self):
        for _ in range(5):
            self.selected_cards.append(self.all_cards.draw())

    # ★ プレイヤー用交換
    def change_card(self):
        print("現在の手札:", self.selected_cards)

        index = int(input(
            f"何番目のカードを捨てる？（1〜5）: "
            f"1={self.selected_cards[0]}, "
            f"2={self.selected_cards[1]}, "
            f"3={self.selected_cards[2]}, "
            f"4={self.selected_cards[3]}, "
            f"5={self.selected_cards[4]} "
        )) - 1

        if index < 0 or index > 4:  #範囲外なら交換しない
            print("無効な入力です。交換しません")
            return

        removed = self.selected_cards.pop(index)
        print("捨てたカード:", removed)

        new_card = self.all_cards.draw()
        self.selected_cards.append(new_card)

        print("引いたカード:", new_card)
        print("交換後の手札:", self.selected_cards)

    # ★ 親が手札を確認して役の弱い場合(One Pare/Two Pare)手札を交換
    def cpu_change_card(self, round_num):
      judge = card_judge(self.selected_cards)
      hand, score = judge.judge()

      values = judge.values
      suits = judge.suits
      counts = Counter(values)

      card_value_pairs = list(zip(self.selected_cards, values))

    # ----------------------
    # ① これ以上強くなりにくいので完成役はキープ
    # ----------------------
      if score >= 6:
        print(f"親は{round_num}回目で交換しませんでした（完成役キープ）")
        return

    # ----------------------
    # ② フラッシュに近い（最優先）
    # 同じマークの枚数を数える:
    # 同じマークが4枚ある → フラッシュ一歩手前
    # ----------------------
      suit_counts = Counter(suits)
      for suit, c in suit_counts.items():
        if c == 4:
            for i, card in enumerate(self.selected_cards):
                if card[0] != suit:
                    self.selected_cards[i] = self.all_cards.draw()
            print(f"親は{round_num}回目でフラッシュ狙い交換")
            return

    # ----------------------
    # ③ ストレートに近い
    # 重複を除いて並べる
    # 「連続してる部分」を残す
    # ----------------------
      unique_values = sorted(set(values))
      for i in range(len(unique_values) - 3):
        if unique_values[i+3] - unique_values[i] == 3: #例5,6,7,8 → 差が3 → 4連続:ストレート一歩手前
            low = unique_values[i]
            high = unique_values[i+3]

            for j, (card, v) in enumerate(card_value_pairs):
                if v < low or v > high: #範囲外のカードを交換
                    self.selected_cards[j] = self.all_cards.draw()
            print(f"親は{round_num}回目でストレート狙い交換")
            return

    # ----------------------
    # ④ スリーカード強化
    # 同じ数字ある状態:異なるものを交換
    # ----------------------
      if score == 3:
        triple = [v for v, c in counts.items() if c == 3]
        for i, (card, v) in enumerate(card_value_pairs):
            if v not in triple:
                self.selected_cards[i] = self.all_cards.draw()
        print(f"親は{round_num}回目でスリーカード強化交換")
        return

    # ----------------------
    # ⑤ ツーペア
    # ペア2つあることを判定: ペア以外（1枚）を交換
    # ----------------------
      if score == 2:
        pairs = [v for v, c in counts.items() if c == 2]
        for i, (card, v) in enumerate(card_value_pairs):
            if v not in pairs:
                self.selected_cards[i] = self.all_cards.draw()
        print(f"親は{round_num}回目でフルハウス狙い交換")
        return

    # ----------------------
    # ⑥ ワンペア
    # ペア1つあることを判定: ペア以外を交換候補とする
    # ----------------------
      if score == 1:
        pair = [v for v, c in counts.items() if c == 2]
        for i, (card, v) in enumerate(card_value_pairs):
            if v not in pair:
                self.selected_cards[i] = self.all_cards.draw()
        print(f"親は{round_num}回目でスリーカード狙い交換")
        return

    # ----------------------
    # ⑦ ハイカード
    # 一番弱いカードを判定して交換対象とする
    # ----------------------
      if score == 0:
        min_value = min(values)
        for i, (card, v) in enumerate(card_value_pairs):
            if v == min_value:
                self.selected_cards[i] = self.all_cards.draw()
                break
        print(f"親は{round_num}回目で弱いカード交換")

# ----------------------------
# 役判定
# ----------------------------
class card_judge:
    def __init__(self, selected_cards):
        self.selected_cards = selected_cards
        #カードを数字部分と絵札部分に分解
        self.suits = [c[0] for c in selected_cards]
        self.ranks = [c[1:] for c in selected_cards]
        #数値化A=14, K=13…など
        rank_dict = {
            "2":2,"3":3,"4":4,"5":5,"6":6,"7":7,
            "8":8,"9":9,"10":10,"J":11,"Q":12,"K":13,"A":14
        }

        self.values = sorted([rank_dict[r] for r in self.ranks])

    def is_flush(self):
        #フラッシュ判定(全部同じマークであるか)
        return len(set(self.suits)) == 1

    def is_straight(self):
        #ストレート判定(連続しているか)
        v = self.values
        return v == list(range(v[0], v[0]+5))

    def judge(self):
        #出現回数で判定([2,2,1] → ツーペア, [3,1,1] → スリーカードなど)
        counts = sorted(Counter(self.values).values(), reverse=True)

        if self.is_flush() and self.is_straight():
            return "ストレートフラッシュ", 8
        elif counts == [4,1]:
            return "フォーカード", 7
        elif counts == [3,2]:
            return "フルハウス", 6
        elif self.is_flush():
            return "フラッシュ", 5
        elif self.is_straight():
            return "ストレート", 4
        elif counts == [3,1,1]:
            return "スリーカード", 3
        elif counts == [2,2,1]:
            return "ツーペア", 2
        elif counts == [2,1,1,1]:
            return "ワンペア", 1
        else:
            return "ハイカード", 0


# -----------------------
# ゲーム開始
# -----------------------

#初期化
all_cards = t_cards()
player1 = player(all_cards)
cpu = player(all_cards)

#ここでカード配る
player1.card_selection()
cpu.card_selection()

print("あなたのカード:", player1.selected_cards)

# --- 交換処理（最大2回）---
for i in range(1, 3):
  while True: #True（真）である限りループする
    decision = input(f"{i}回目：交換するなら1、しないなら2: ")

    if decision == "1":
      player1.change_card()
      break #正しい入力（1 or 2）のときだけループ終了
    elif decision == "2":
      print("カードを交換しません")
      break #正しい入力（1 or 2）のときだけループ終了
    else: #1 or 2以外のときは再入力を要求
      print("入力した値は無効です。1(交換する)か2(交換しない)を入力してください")

  cpu.cpu_change_card(i)

print("最終手札:", player1.selected_cards)
print("\n親のカード:", cpu.selected_cards)

# -----------------------
# 判定
# -----------------------
player_judge = card_judge(player1.selected_cards)
cpu_judge = card_judge(cpu.selected_cards)

player_hand, player_score = player_judge.judge()
cpu_hand, cpu_score = cpu_judge.judge()

print("\nあなたの役:", player_hand)
print("親の役:", cpu_hand)

# -----------------------
# 勝敗
# -----------------------
if player_score > cpu_score:
    print("あなたの勝ち！")
elif player_score < cpu_score:
    print("親の勝ち！")
else:
    print("引き分け！")

あなたのカード: ['♠7', '♠10', '♥Q', '♦7', '♦J']
1回目：交換するなら1、しないなら2: 2
カードを交換しません
親は1回目で弱いカード交換
2回目：交換するなら1、しないなら2: 2
カードを交換しません
親は2回目でスリーカード狙い交換
最終手札: ['♠7', '♠10', '♥Q', '♦7', '♦J']

親のカード: ['♠9', '♥8', '♦2', '♠3', '♥6']

あなたの役: ワンペア
親の役: ハイカード
あなたの勝ち！
